## 2.2 Tokenizing Text

In [1]:
import urllib.request

url = ("https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt")

file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x2ac50763d90>)

In [2]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [3]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [4]:
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [5]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [6]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [7]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


In [8]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## 2.3 Tokens to Token IDs

In [9]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [10]:
vocab = {token:integer for integer,token in enumerate(all_words)}
# i have not included the code of printing the first few entries to check if vocab is correct or not.

In [11]:
from tokenizer import SimpleTokenizerV1

In [12]:
tokenizer = SimpleTokenizerV1(vocab)
text = """Hi! My name is Annant Pathak. I'm a second Year B.Tech student in IIT Hyderabad"""
ids = tokenizer.encode(text)
print(ids)
# Keyerror Hi -- It means the keyword hi was not in our vocab, so this normal tokenizer can't encode this as it doesn't
# Have the given ID for this text. Thats why we need large datasets to train LLMs. or other things like
# <unk>, <end of text>

KeyError: 'Hi'

In [13]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
 Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [14]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


## 2.4 Adding special context tokens

In [15]:
# We're adding two special tokens end of text and unk
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}
print(len(vocab.items()))

1132


In [16]:
# Concept of tokenizer 2 is similar to tokenizer 1, we're just adding the logic of handling unknown words.
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [item if item in self.str_to_int
                            else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [17]:
text1 = "Hi i'm Annant Pathak."
text2 = "IIT hyderabad is sixth in NIRF."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hi i'm Annant Pathak. <|endoftext|> IIT hyderabad is sixth in NIRF.


In [18]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))
#always the unknown text is kept at last of vocab, so you can see so many id as 1131 it means they are unknown.

[1131, 1131, 2, 1131, 1131, 1131, 7, 1130, 1131, 1131, 584, 1131, 568, 1131, 7]


In [19]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|> <|unk|>' <|unk|> <|unk|> <|unk|>. <|endoftext|> <|unk|> <|unk|> is <|unk|> in <|unk|>.


## 2.5 Byte Pair Encoding (BPE)

In [20]:
# BPE is complex so we won't code it. We'll import it from Tiktoken library.
!pip install tiktoken
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


tiktoken version: 0.12.0


In [21]:
tokenizer = tiktoken.get_encoding("gpt2")

In [22]:
text = (
 "Hi, i'm Annant Pathak <|endoftext|> IIT hyderabad is the sixth in NIRF."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[17250, 11, 1312, 1101, 5506, 415, 10644, 461, 220, 50256, 314, 2043, 2537, 1082, 17325, 318, 262, 11695, 287, 399, 4663, 37, 13]


In [23]:
strings = tokenizer.decode(integers)
print(strings)

Hi, i'm Annant Pathak <|endoftext|> IIT hyderabad is the sixth in NIRF.
